# A–D 페어 선정 · 청산 · 손절 144개 실험

A → B → C → D, 각 모델 36개를 순서대로 실행합니다. 트레일링 없이 청산 `0 / +0.5`, 추정 `12 / 24개월`, 손절 `3 / 3.5 / … / 7σ`를 비교합니다.

청산 목표는 진입 방향에 맞춰 부호가 반대가 됩니다. `exit_z=0.5` 한 설정이 양의 Z 진입은 +0.5, 음의 Z 진입은 −0.5에서 청산합니다. 두 방향을 별도 실험으로 세지 않습니다. 진입·청산 신호는 다음 거래일 종가에 체결합니다.

규칙과 출력: [실험 설명](../docs/selection_study.md). 이미 열려 있던 노트북에서는 **커널을 다시 시작**해 새 코드를 불러오세요. 실데이터 전체 탐색은 오래 걸릴 수 있습니다.

**합병 발표 정책이 적용된 새 실험입니다.** 커널을 재시작하고 이 노트북을 처음부터 실행하세요. 이전 216개 실험 결과는 보존하며 새 폴더에 저장합니다. 기본 기간은 최근 실제 실행과 같은 2020–2025년입니다.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "run_backtest.py").is_file() and (p / "src/sp500_pairs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from sp500_pairs import load_config, load_data, build_study_plan, run_selection_study
print("프로젝트:", ROOT)


## 1. 내 실행 설정

`DATA_DIR`에는 원래 사용하던 실제 가격·구성종목·섹터 자료 폴더를 지정하세요. 개인 경로는 공유 전에 정리하세요.

`MODELS=["A"]`로 A만 먼저 돌리고, 이후 전체 모델 또는 B/C/D로 바꿔 같은 폴더에서 이어갈 수도 있습니다. 처음부터 전체 모델이면 A 36개 → B 36개 → C 36개 → D 36개입니다.

`PARAMETERS`는 공통 조건입니다. 추정개월·청산·손절·필터 사용 여부는 144개 실험 계획이 지정합니다.

In [ ]:
CONFIG_FILE = "configs/selection_study.toml"
DATA_DIR = Path.home() / "Desktop" / "pair"  # 원래 사용하던 데이터 폴더로 수정
OUTPUT_DIR = ROOT / "results" / "selection_study_144_no_concentration_caps"
MODELS = ["A", "B", "C", "D"]
RETRY_FAILED = False
OUTPUT_PROFILE = "compact"  # 처음부터 "full"이면 페어별 일별 원장/그림도 저장

PARAMETERS = {
    "selection.max_per_sector": 0,  # 섹터당 페어 수 제한 없음
    "selection.max_per_stock": 0,  # 종목당 페어 수 제한 없음
    "schedule.start": "2020-01-01",
    "schedule.end": "2025-12-31",
    "schedule.trading_months": 6,
    "run.max_windows": 0,  # 0: 전체. 점검용 1이면 OUTPUT_DIR도 별도로 지정
    "formation.min_return_correlation": 0.5,
    "pretrade.months": 12,
    "pretrade.min_entries": 3,
    "pretrade.min_sharpe": 0.0,
    "corporate_actions.enabled": True,
    "corporate_actions.events_file": "reference_data/merger_events.csv",
}
cfg = load_config(ROOT / CONFIG_FILE).with_updates(PARAMETERS)
plan = build_study_plan()
print("전체 계획:", len(plan), "개; 선택한 모델:", MODELS)
display(plan.groupby("model", sort=False).size().rename("실험 수").to_frame())
display(plan.head(12))


## 2. 입력 자료 확인

C/D는 거래 시작 전 12개월 동안 사전 모의거래를 하며, 6개월 블록마다 그 이전 F개월로 회귀를 추정합니다. F24·2020년 시작은 2017년부터 자료가 필요합니다. 선정용 모의거래 규칙은 진입 2·청산 0·손절 3.5σ로 고정하여, 청산/손절 실험값을 바꿔도 선정 페어가 달라지지 않게 합니다.

기존 진입 상한 규칙도 유지하므로 진입은 `2 ≤ |Z| < 손절선`입니다. 손절 탐색은 진입 상한도 함께 바꿉니다. 보유 중 가격 결측은 임의로 채우지 않고 실패로 기록합니다.

합병 발표가 알려지면 신규 진입을 취소하고 보유 페어 양쪽을 다음 거래 가능한 종가에 청산합니다. 발표 시각을 확인하지 못한 일정은 다음 거래일부터 알려진 것으로 반영합니다. 일정은 수동으로 확인한 35건이며 전체 기업행동을 망라하지 않습니다. [일정·출처·시점 규칙](../reference_data/README.md)을 확인하세요.


가격이 2005-09-27부터라면, A–D를 동일한 1월/7월 반기 구간으로 비교할 때 과거 자료 길이를 충족하는 첫 시작일은 **2009-01-01**입니다. C/D의 F24는 추정 24개월 + 사전 모의거래 12개월로 총 36개월이 필요합니다. 아래 표는 현재 `cfg` 기준이며 시작일을 자동 변경하지 않습니다. 날짜를 바꾸었으면 `cfg = ...with_updates(PARAMETERS)`가 있는 설정 셀부터 다시 실행하세요.


In [ ]:
data = load_data(cfg, data_dir=DATA_DIR)
print("자료:", data.metadata.get("mode"))
print("가격 기간:", data.prices.index.min().date(), "~", data.prices.index.max().date())
print("가격 종목 수:", len(data.prices.columns))
print("평가:", cfg["evaluation"]["active_split"])
if data.metadata.get("mode") == "synthetic":
    print("가상 데이터: 투자 성과가 아닌 소프트웨어 실행 점검입니다.")

from sp500_pairs.corporate_actions import calendar_for
mergers = calendar_for(data, cfg)
print("합병 정책:", cfg["corporate_actions"]["enabled"], "| 등록 사건:", len(mergers.events))
display(pd.DataFrame(mergers.records())[["ticker", "announcement_date", "timing", "release_date", "source"]])

# 실행 중인 cfg와 데이터가 각 모델의 과거 자료 요구를 충족하는지 먼저 확인합니다.
from sp500_pairs.study import study_history_requirements
print("실행 설정 기간:", cfg.date_range()[0].date(), "~", cfg.date_range()[1].date())
history = study_history_requirements(cfg, data.prices.index, models=MODELS)
display(history.rename(columns={
    "model": "모델", "formation_months": "추정 개월", "pretrade_months": "사전거래 개월",
    "history_months": "필요 과거 개월", "first_window": "첫 거래 구간",
    "required_price_start": "필요 가격 시작", "actual_price_start": "실제 가격 시작",
    "history_available": "과거 기간 충족", "suggested_first_window": "기간 충족 첫 구간",
}))
if not history["history_available"].all():
    print("설정 셀에서 시작일을", history["suggested_first_window"].max(),
          "이후로 바꾸거나 더 오래된 가격을 확보하세요. cfg 생성 셀도 다시 실행해야 합니다.")


## 3. 실행 / 이어서 실행

이 셀은 실제 실험을 시작합니다. 완료된 실험은 건너뛰고 중단된 실험은 새 attempt 폴더에서 다시 실행합니다. 실패한 실험은 기록 후 다음 조합으로 넘어갑니다. 실패 재시도는 위의 `RETRY_FAILED=True`로 지정하세요.

설정·가격·합병 일정·코드·패키지가 바뀌었다면 `OUTPUT_DIR`을 새 폴더로 바꾸세요. 기존 결과와 섞지 않습니다. 전체 구간 결과는 탐색용이며 최종 test 기간에서는 실행할 수 없습니다.

In [ ]:
comparison, study_folder = run_selection_study(
    cfg, data=data, output_dir=OUTPUT_DIR, models=MODELS,
    resume=True, retry_failed=RETRY_FAILED, output_profile=OUTPUT_PROFILE,
)
print("전체 비교표:", study_folder / "comparison.csv")
display(comparison.groupby(["model", "status"], sort=False).size().rename("실험 수").to_frame())


## 4. 완료 성과와 실패 확인

성과 비교는 완료된 실험만 사용합니다. 144개가 모두 성공해야 전체 완료입니다. 등록된 합병 발표에는 신규 진입 차단·보유 페어 청산을 적용합니다. 일정표 누락이나 일반 가격 결측이 남으면 실패 원인을 별도로 확인해야 합니다.

In [ ]:
completed = comparison[comparison["status"] == "complete"].copy()
failed = comparison[comparison["status"] == "failed"]
columns = ["name", "model", "formation_months", "exit_z", "stop_z", "sharpe",
           "annualized_return", "max_drawdown", "completed_trades", "merger_announcement_exits", "output_directory"]
if not completed.empty:
    display(completed[[c for c in columns if c in completed]].sort_values("sharpe", ascending=False).head(20))
if not failed.empty:
    display(failed[["name", "error", "output_directory"]])
print("완료:", len(completed), "/ 144, 실패:", len(failed))
